##  Content-Based Filtering

In [1]:
import os, re
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")
PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/content_based"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Ready!")

Ready!


In [2]:
master = pd.read_csv("data/processed/movies_master.csv")
for col in ["overview","tagline","genres","keywords","director","cast"]:
    master[col] = master[col].fillna("")

def build_weighted_soup(row):
    def clean(text):
        text = str(text).lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    overview = clean(row["overview"])
    tagline  = clean(row["tagline"])
    genres   = " ".join([clean(row["genres"].replace("|", " "))] * 4)
    director = " ".join([clean(row["director"].replace(" ", "_"))] * 3)
    cast_list = [clean(c.replace(" ", "_")) for c in str(row["cast"]).split("|")[:3]]
    cast     = " ".join(cast_list * 2)
    keywords = clean(row["keywords"].replace("|", " "))
    return f"{overview} {tagline} {genres} {director} {cast} {keywords}"

print("Building improved soup...")
master["content_soup"] = master.apply(build_weighted_soup, axis=1)
df = master[["movie_id","title","overview","content_soup","poster_path"]].copy()
df.to_csv(f"{PROCESSED_DIR}/content_based.csv", index=False)
print(f"Done! {len(df)} movies")

Building improved soup...
Done! 9988 movies


In [3]:
print("Building TF-IDF matrix...")
tfidf = TfidfVectorizer(
    max_features = 20000,
    min_df       = 2,
    max_df       = 0.8,
    ngram_range  = (1, 2),
    sublinear_tf = True
)
tfidf_matrix = tfidf.fit_transform(df["content_soup"])
print(f"Matrix shape: {tfidf_matrix.shape}")

Building TF-IDF matrix...
Matrix shape: (9988, 20000)


In [4]:
print("Computing cosine similarity... (1-2 min)")
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Shape: {cosine_sim.shape}")
print(f"Memory: {cosine_sim.nbytes/1024/1024:.1f} MB")

Computing cosine similarity... (1-2 min)
Shape: (9988, 9988)
Memory: 761.1 MB


In [5]:
indices = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()

def get_recommendations(title: str, n: int = 10) -> pd.DataFrame:
    title_lower = title.lower().strip()
    
    if title_lower not in indices:
        matches = [t for t in indices.index if title_lower in t]
        if not matches:
            print(f"'{title}' not found!")
            return pd.DataFrame()
        title_lower = matches[0]
        print(f"Using: '{title_lower}'")
    
    idx        = indices[title_lower]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    movie_idx  = [i[0] for i in sim_scores]
    scores     = [round(i[1], 4) for i in sim_scores]
    
    result = df.iloc[movie_idx][["movie_id","title","overview"]].copy()
    result["similarity_score"] = scores
    return result.reset_index(drop=True)

print("get_recommendations() is ready!")

get_recommendations() is ready!


In [6]:
test_movies = ["Inception", "The Dark Knight", "Toy Story", "The Godfather", "Interstellar"]

print("=== Quality Check ===\n")
for movie in test_movies:
    recs = get_recommendations(movie, n=5)
    if not recs.empty:
        print(f"[{movie}]")
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} score: {row['similarity_score']:.4f}")
        print()

=== Quality Check ===

[Inception]
  → Interstellar                             score: 0.2426
  → Tenet                                    score: 0.2388
  → The Prestige                             score: 0.2218
  → Following                                score: 0.2216
  → Don Jon                                  score: 0.2212

[The Dark Knight]
  → The Dark Knight Rises                    score: 0.4077
  → Batman Begins                            score: 0.3508
  → The Prestige                             score: 0.2421
  → Following                                score: 0.2148
  → Insomnia                                 score: 0.1983

[Toy Story]
  → Toy Story 2                              score: 0.4605
  → Toy Story 4                              score: 0.4032
  → Toy Story 3                              score: 0.3817
  → Hawaiian Vacation                        score: 0.3737
  → Small Fry                                score: 0.3321

[The Godfather]
  → The Godfather Part II      

In [7]:
print("Saving model...")
joblib.dump(tfidf, f"{MODELS_DIR}/tfidf_vectorizer.pkl")
np.save(f"{MODELS_DIR}/cosine_sim.npy", cosine_sim)
indices.to_pickle(f"{MODELS_DIR}/indices.pkl")
df.to_csv(f"{MODELS_DIR}/content_df.csv", index=False)

print("Saved:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024 / 1024
    print(f"  {f:35s} {size:.1f} MB")

Saving model...
Saved:
  content_df.csv                      9.2 MB
  cosine_sim.npy                      761.1 MB
  indices.pkl                         0.3 MB
  tfidf_vectorizer.pkl                6.5 MB
